#### Using Proto-ML [1] to analyse the drug reviews dataset [2]

[1] https://github.com/yx131/proto-lm/tree/main

[2] https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data

### Global imports

In [1]:
import argparse
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
import datasets

### Global Params

In [2]:
train_the_model = False

In [3]:
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import os
import nltk
nltk.download('vader_lexicon')

def add_sentiment_features(df, text_column):
    """
    Adds sentiment analysis features to the dataset using VADER.

    Args:
        df (pd.DataFrame): The input DataFrame containing the text data.
        text_column (str): The name of the column containing the text.

    Returns:
        pd.DataFrame: The updated DataFrame with sentiment features.
    """
    # Initialize the VADER sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()

    # Compute sentiment scores for each text
    sentiment_scores = df[text_column].fillna("").apply(analyzer.polarity_scores)

    # Convert the sentiment scores into a DataFrame
    sentiment_df = pd.json_normalize(sentiment_scores)

    # Rename the sentiment columns for clarity
    sentiment_df.rename(columns={
        'neg': 'sentiment_neg',
        'neu': 'sentiment_neu',
        'pos': 'sentiment_pos',
        'compound': 'sentiment_compound'
    }, inplace=True)

    # Join the sentiment features with the original DataFrame
    df = df.join(sentiment_df.set_index(df.index))
    return df

# Process all datasets
data_files = {
    "train": "../Data/drug_review_train_clean.csv",
    "validation": "../Data/drug_review_validation_clean.csv",
    "test": "../Data/drug_review_test_clean.csv"
}

for split, file_path in data_files.items():
    output_path = file_path.replace("_clean.csv", "_with_sentiment.csv")
    
    # Check if the output file already exists
    if os.path.exists(output_path):
        print(f"Skipping {split} dataset as {output_path} already exists.")
        continue

    print(f"Processing {split} dataset...")
    
    # Load the dataset
    data = pd.read_csv(file_path)
    
    # Add sentiment features
    data = add_sentiment_features(data, text_column="review_clean")
    
    # Save the updated dataset
    data.to_csv(output_path, index=False)
    print(f"Sentiment features added and saved to {output_path}")

Skipping train dataset as ../Data/drug_review_train_with_sentiment.csv already exists.
Skipping validation dataset as ../Data/drug_review_validation_with_sentiment.csv already exists.
Skipping test dataset as ../Data/drug_review_test_with_sentiment.csv already exists.


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\bar24\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


### Drug Review Data Class

In [4]:
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd

class sst_datamodule(pl.LightningDataModule):
    loader_columns = [
        "datasets_idx",
        "input_ids",
        "token_type_ids",
        "attention_mask",
        "start_positions",
        "end_positions",
        "labels",
        "sentiment_features",
        "sentiment_neg",
        "sentiment_neu",
        "sentiment_pos",
        "sentiment_compound"
    ]

    def __init__(
            self,
            model_name_or_path: str,
            max_seq_length: int = 128,
            train_batch_size: int = 32,
            eval_batch_size: int = 32,
            dataset=None,  # Accept pre-loaded dataset
            **kwargs,
    ):
        super().__init__()
        self.model_name_or_path = model_name_or_path
        self.max_seq_length = max_seq_length
        self.train_batch_size = train_batch_size
        self.eval_batch_size = eval_batch_size
        self.dataset = dataset

        self.text_fields = ['review_clean']
        self.num_labels = 10  # Number of classes in drug review dataset, a user can rate from 1 to 10
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name_or_path, use_fast=True)

    def load_dataset_locally(self):
        from datasets import load_dataset
        data_files = {
            "train": "../Data/drug_review_train_with_sentiment.csv",
            "validation": "../Data/drug_review_validation_with_sentiment.csv",
            "test": "../Data/drug_review_test_with_sentiment.csv"
        }
        self.dataset = load_dataset("csv", data_files=data_files)

    def setup(self, stage: str = None):
        if self.dataset is None:
            print("Loading dataset from local files...")
            self.load_dataset_locally()

        for split in self.dataset.keys():
            print(f'split is: {split}')
            

            if "rating" in self.dataset[split].column_names:
                self.dataset[split] = self.dataset[split].rename_column("rating", "label")


            self.dataset[split] = self.dataset[split].map(
                self.convert_to_features,
                batched=True,
                # remove_columns=["label", "Unnamed: 0"],
            )
            
            self.columns = [c for c in self.dataset[split].column_names if c in self.loader_columns]
            print(f'self.columns: {self.columns}')
            self.dataset[split].set_format(type="torch", columns=self.columns)

        self.eval_splits = [x for x in self.dataset.keys() if "validation" in x]

    def train_dataloader(self):
        print(f'returned self batch size: {self.train_batch_size}')
        return DataLoader(self.dataset["train"], batch_size=self.train_batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.dataset["validation"], batch_size=self.eval_batch_size)

    def test_dataloader(self):
        return DataLoader(self.dataset["test"], batch_size=self.eval_batch_size)

    def convert_to_features(self, example_batch, indices=None):
        if len(self.text_fields) > 1:
            texts_or_text_pairs = list(zip(example_batch[self.text_fields[0]], example_batch[self.text_fields[1]]))
        else:
            texts_or_text_pairs = example_batch[self.text_fields[0]]

        features = self.tokenizer.batch_encode_plus(
            texts_or_text_pairs,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=self.max_seq_length
        )

        # features["labels"] = [1 if int(label) >= 7 else 0 for label in example_batch["label"]]
        # Convert labels to zero-indexed (1-10 becomes 0-9)
        features["labels"] = [int(label) - 1 for label in example_batch["label"]]
        # # Add sentiment features
        # features["sentiment_neg"] = torch.tensor(example_batch["sentiment_neg"]).unsqueeze(1)
        # features["sentiment_neu"] = torch.tensor(example_batch["sentiment_neu"]).unsqueeze(1)
        # features["sentiment_pos"] = torch.tensor(example_batch["sentiment_pos"]).unsqueeze(1)
        # features["sentiment_compound"] = torch.tensor(example_batch["sentiment_compound"]).unsqueeze(1)
        # Combine sentiment features into a single tensor
        sentiment_features = torch.stack([
            torch.tensor(example_batch["sentiment_neg"]),
            torch.tensor(example_batch["sentiment_neu"]),
            torch.tensor(example_batch["sentiment_pos"]),
            torch.tensor(example_batch["sentiment_compound"])
        ], dim=1)  # Shape: (batch_size, 4)
        features["sentiment_features"] = sentiment_features
        return features

In [5]:
from transformers import AutoConfig

# Parameters for Proto-LM training on the drug review dataset
model_name = 'bert-base-uncased'  # Backbone LLM model to load
# Maximum sentence length to pad/truncate to
args = {
    'model_name': model_name,        # backbone LLM model to load
    'max_seq_length': 100,                # maximum sentence length to pad/truncate to
    'num_prototypes': 100,               # number of prototypes to train
    'hidden_shape': 1024,                # hidden shape of each prototype, should match LLM output
    'num_classes': 10,                    # number of output classes
    'cohsep_ratio': 0.5,                 # ratio of prototypes in class to push/pull
    'lambda0': 0.5,                      # lambda0 in loss
    'lr': 3e-4,                          # initial learning rate
    'proto_training_weights': 1,         # whether to train prototype weights (1=True, 0=False)
    'batch_size': 32,                   # batch size for dataloader
    'logger_dir': 'tb_logs',             # directory for the logger to store training details
    'checkpoint_dir': 'ckpt_dir',        # directory to store checkpoints
    'config_subdir': 'config_subdir',    # subdirectory for checkpoints of a certain config
    'max_epochs': 1,                     # number of epochs to train
    'num_gpu': 1,                        # number of gpus to train on
    'load_model': model_name,  # path to load a pretrained model, if any
}



# Dynamically fetch hidden size from the model configuration
config = AutoConfig.from_pretrained(args['model_name'])
hidden_size = config.hidden_size  # Dynamically get the hidden size (768 for bert-base-uncased)
args['hidden_shape'] = hidden_size

print(f'args: {args}')


# get data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)
drug_review_dm.setup(stage='fit')

args: {'model_name': 'bert-base-uncased', 'max_seq_length': 100, 'num_prototypes': 100, 'hidden_shape': 768, 'num_classes': 10, 'cohsep_ratio': 0.5, 'lambda0': 0.5, 'lr': 0.0003, 'proto_training_weights': 1, 'batch_size': 32, 'logger_dir': 'tb_logs', 'checkpoint_dir': 'ckpt_dir', 'config_subdir': 'config_subdir', 'max_epochs': 1, 'num_gpu': 1, 'load_model': 'bert-base-uncased'}
Loading dataset from local files...
split is: train
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: validation
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: test
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']


In [6]:

print(f"loading a model: {args['load_model']}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'], ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

proto = proto_lm(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights']),
)

loading a model: bert-base-uncased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Train

In [7]:
if train_the_model:
    # get training utilities like logger and checkpoints
    from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
    from pytorch_lightning.callbacks import ModelCheckpoint

    tb_logger = TensorBoardLogger(f"{args['logger_dir']}", name="drug_review_tensorboard_logs")
    ckpt_path = f"{args['checkpoint_dir']}/{args['config_subdir']}"
    checkpoint_callback = ModelCheckpoint(
        dirpath=ckpt_path,
        monitor='val_loss',
        save_top_k=3,
        filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}"
    )

    # get trainer object
    trainer = pl.Trainer(
        max_epochs=args['max_epochs'],
        accelerator="auto",
        devices=1,  # or just remove this line for auto
        logger=tb_logger,
        callbacks=[checkpoint_callback],
        log_every_n_steps=50  # Log every 50 steps instead of every step
    )

    trainer.fit(proto, datamodule=drug_review_dm)

    # Optionally test or save misclassified
    # trainer.test(proto, datamodule=drug_review_dm, ckpt_path=args['load_model'])
    # torch.save(proto.misclassified, 'drug_review_logs/misclassed.pt')

### Save the trained model

In [8]:
if train_the_model:
# Save the final model weights
    torch.save(proto.state_dict(), "final_proto_model.pt")
    print("Training complete. Model weights saved as 'final_proto_model.pt'.")

### Load the trained model

In [9]:
# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proto.load_state_dict(torch.load("final_proto_model.pt", weights_only=False))  # Load the state dictionary into the model
print("Model loaded successfully from 'final_proto_model.pt'.")

proto.to(device)  # Move the model to the appropriate device
print(f"Model moved to device: {device}")

Model loaded successfully from 'final_proto_model.pt'.
Model moved to device: cuda


### Calc Quantus Metrics

In [10]:
def forward_func(embeddings, attention_mask=None, token_type_ids=None, sentiment_features=None):
    output = proto(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
        sentiment_features=sentiment_features
    )
    return output['logits']  # Extract logits from the output dictionary

In [11]:
from torch.utils.data import DataLoader

print("Preparing test data loader...")
test_loader = drug_review_dm.test_dataloader()
batch = next(iter(test_loader))
input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)
token_type_ids = batch.get('token_type_ids', None)
if token_type_ids is not None:
    token_type_ids = token_type_ids.to(device)
labels = batch['labels'].to(device)

Preparing test data loader...


In [12]:
from captum.attr import IntegratedGradients
import numpy as np

print("Generating saliency maps using Integrated Gradients...")
ig = IntegratedGradients(forward_func)

# Get embeddings from embedding layer
input_embeddings = proto.LLM.embeddings(input_ids)  # Shape: [batch_size, seq_len, hidden_dim]
input_embeddings.requires_grad_()  # Needed for IG

# Now call IntegratedGradients
saliency = ig.attribute(
    inputs=input_embeddings,
    additional_forward_args=(attention_mask, token_type_ids, batch['sentiment_features'].to(device)),
    target=0,
    n_steps=10,
    internal_batch_size=4
)

sentiment_features = batch['sentiment_features'].to(device)

# Combine inputs into a single tensor for Quantus
x_batch = torch.cat(
    [input_ids.float(), attention_mask.float(), sentiment_features.float()], dim=-1
).detach().cpu().numpy()  # Move to CPU if Quantus requires it

# Move labels to CPU as well
y_batch = labels.detach().cpu().numpy()

# Convert saliency to numpy
explanations = saliency.detach().cpu().numpy()

# Aggregate saliency maps to match the dimensionality of x_batch
explanations = explanations.sum(axis=-1)  # Shape: [batch_size, seq_len]

# Pad explanations to match the shape of x_batch
seq_len = input_ids.shape[1]  # Length of the input sequence
total_features = x_batch.shape[1]  # Total features in x_batch
padding = total_features - seq_len  # Calculate the padding needed

# Pad the saliency map with zeros along the last axis
explanations_padded = np.pad(explanations, ((0, 0), (0, padding)), mode='constant')  # Shape: [batch_size, total_features]

# # Use explanations_padded as the saliency map for Quantus
# complexity_metric = quantus.Complexity()
# complexity_score = complexity_metric(
#     model=proto,
#     x_batch=x_batch,
#     y_batch=y_batch,
#     a_batch=explanations_padded,  # Use the padded saliency map
#     device=device
# )
# print(f"Complexity Score: {complexity_score}")

Generating saliency maps using Integrated Gradients...


c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\captum\attr\_utils\batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 32 equal to the number of examples.
  warnings.warn(


In [13]:
import quantus

# Complexity Metric
complexity_metric = quantus.Complexity(return_aggregate=True)
complexity_score = complexity_metric(
    model=proto,
    x_batch=x_batch,
    y_batch=y_batch,
    a_batch=explanations_padded,  # Saliency maps
    device=device
)


complexity_score = complexity_score[0]
print(f"Complexity Score: {complexity_score:.4f}")

Complexity Score: 3.1788


In [14]:
# import torch.nn as nn

# # Define a wrapper for the proto_lm model
# class ProtoLMWrapper(nn.Module):
#     def __init__(self, model):
#         super(ProtoLMWrapper, self).__init__()
#         self.model = model

#     def forward(self, x):
#         # Extract components from x (assuming x is a tensor with all features concatenated)
#         # Adjust the slicing based on your input structure
#         seq_length = self.model.max_seq_length
#         sentiment_dim = 4  # Number of sentiment features

#         input_ids = x[:, :seq_length].long()
#         attention_mask = x[:, seq_length:2 * seq_length].long()
#         sentiment_features = x[:, 2 * seq_length:].float()

#         # Call the proto_lm model with the correct arguments
#         outputs = self.model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             sentiment_features=sentiment_features
#         )
#         return outputs["logits"]  # Return logits as expected by Quantus

# # Wrap the proto_lm model
# wrapped_model = ProtoLMWrapper(proto)

# # Ensure the model is in evaluation mode
# wrapped_model.eval()

# # Define subset_size based on the number of features in x_batch
# total_features = x_batch.shape[1]  # Total features in x_batch
# subset_size = max(1, min(10, total_features // 2))  # Ensure subset_size is at least 1 and smaller than total_features

# # Pass the wrapped model to Quantus
# faithfulness_metric = quantus.FaithfulnessCorrelation(subset_size=subset_size)
# faithfulness_score = faithfulness_metric(
#     model=wrapped_model,
#     x_batch=x_batch,
#     y_batch=y_batch,
#     a_batch=explanations_padded,  # Saliency maps
#     device=device
# )
# print(f"Faithfulness Score: {faithfulness_score}")

In [15]:
# Define a wrapper for the proto_lm model
import torch.nn as nn


class ModelWrapper(nn.Module):
    def __init__(self, model):
        super(ModelWrapper, self).__init__()
        self.model = model

    def forward(self, input_ids=None, attention_mask=None, sentiment_features=None, inputs_embeds=None):
        device = next(self.model.parameters()).device

        # Debugging: Print input shapes
        # print(f"ModelWrapper.forward() called with:")
        # print(f"  input_ids shape: {input_ids.shape if input_ids is not None else None}")
        # print(f"  attention_mask shape: {attention_mask.shape if attention_mask is not None else None}")
        # print(f"  sentiment_features shape: {sentiment_features.shape if sentiment_features is not None else None}")
        # print(f"  inputs_embeds shape: {inputs_embeds.shape if inputs_embeds is not None else None}")

        # Ensure inputs_embeds is passed correctly
        if inputs_embeds is not None:
            inputs_embeds = inputs_embeds.to(device)
            input_ids = None  # Avoid passing input_ids when using embeddings
        elif input_ids is not None:
            input_ids = input_ids.to(device).long()  # Ensure input_ids is in the correct shape

        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        if sentiment_features is not None:
            sentiment_features = sentiment_features.to(device)

        # Pass inputs to the model
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            inputs_embeds=inputs_embeds
        )

        # Debugging: Print output shapes
        # print(f"  logits shape: {outputs['logits'].shape}")
        return outputs['logits']
    
# Wrap the proto model
wrapped_model = ModelWrapper(proto)
wrapped_model.eval()  # Set the model to evaluation mode

ModelWrapper(
  (model): proto_lm(
    (LLM): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True)
        

In [16]:
from tqdm import tqdm

# Custom aggregation function to debug NaN values
def custom_aggregation(scores_list):
    print("Layer-wise MPRT Scores:")
    valid_scores = []
    with tqdm(total=len(scores_list), desc="Aggregating Scores") as pbar:
        for score in scores_list:
            print(f"Score: {score}")
            if not np.isnan(score):
                valid_scores.append(score)
            pbar.update(1)  # Update the progress bar

    if not valid_scores:
        print("All scores are NaN! Returning NaN.")
        return np.nan

    mean_score = np.mean(valid_scores)
    print(f"Mean score: {mean_score}")
    return mean_score

# def generate_saliency_map_(model, input_ids, attention_mask, sentiment_features, targets, **kwargs):
#     print(f"generate_saliency_map_ called with:")
#     print(f"  input_ids shape: {input_ids.shape}")
#     print(f"  attention_mask shape: {attention_mask.shape}")
#     print(f"  sentiment_features shape: {sentiment_features.shape}")
#     print(f"  targets shape: {targets.shape}")

#     # Ensure all inputs are torch.Tensor and move them to the correct device
#     device = next(model.parameters()).device
#     if not isinstance(input_ids, torch.Tensor):
#         input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
#     if not isinstance(attention_mask, torch.Tensor):
#         attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)
#     if not isinstance(sentiment_features, torch.Tensor):
#         sentiment_features = torch.tensor(sentiment_features, dtype=torch.float32).to(device)
#     if not isinstance(targets, torch.Tensor):
#         targets = torch.tensor(targets, dtype=torch.long).to(device)

#     # Get embeddings from the model's embedding layer
#     inputs_embeds = model.model.LLM.embeddings(input_ids)
#     inputs_embeds.requires_grad_()  # Ensure gradients can be computed for embeddings

#     # Define a forward function for Integrated Gradients
#     def forward_func(inputs_embeds, attention_mask, sentiment_features):
#         outputs = model(
#             inputs_embeds=inputs_embeds,
#             attention_mask=attention_mask,
#             sentiment_features=sentiment_features
#         )
#         # Debugging: Print logits shape
#         print(f"  logits shape in forward_func: {outputs.shape}")
#         return outputs

#     ig = IntegratedGradients(forward_func)
#     saliency = ig.attribute(
#         inputs=inputs_embeds,  # Use embeddings as the main input
#         additional_forward_args=(attention_mask, sentiment_features),
#         target=targets,
#         internal_batch_size=4  # Process 4 samples at a time
#     )
#     print(f"Saliency map shape: {saliency.shape}")
#     print(f"Saliency map (first 5 values): {saliency.flatten()[:5]}")  # Debugging: Print first 5 values
#     return saliency.detach().cpu().numpy()


def generate_saliency_map_(model, input_ids, attention_mask, sentiment_features, targets, **kwargs):
    # print(f"generate_saliency_map_ called with:")
    # print(f"  input_ids shape: {input_ids.shape}")
    # print(f"  attention_mask shape: {attention_mask.shape}")
    # print(f"  sentiment_features shape: {sentiment_features.shape}")
    # print(f"  targets shape: {targets.shape}")

    # Ensure all inputs are torch.Tensor and move them to the correct device
    device = next(model.parameters()).device
    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.tensor(input_ids, dtype=torch.long).to(device)
    if not isinstance(attention_mask, torch.Tensor):
        attention_mask = torch.tensor(attention_mask, dtype=torch.long).to(device)
    if not isinstance(sentiment_features, torch.Tensor):
        sentiment_features = torch.tensor(sentiment_features, dtype=torch.float32).to(device)
    if not isinstance(targets, torch.Tensor):
        targets = torch.tensor(targets, dtype=torch.long).to(device)

    # Get embeddings from the model's embedding layer
    inputs_embeds = model.model.LLM.embeddings(input_ids)
    inputs_embeds.requires_grad_()  # Ensure gradients can be computed for embeddings

    # Define a forward function for Integrated Gradients
    def forward_func(inputs_embeds, attention_mask, sentiment_features):
        outputs = model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features
        )
        # Debugging: Print logits shape
        # print(f"  logits shape in forward_func: {outputs.shape}")
        return outputs

    ig = IntegratedGradients(forward_func)
    saliency = ig.attribute(
        inputs=inputs_embeds,  # Use embeddings as the main input
        additional_forward_args=(attention_mask, sentiment_features),
        target=targets,
        internal_batch_size=4  # Process 4 samples at a time
    )
    # print(f"Saliency map shape: {saliency.shape}")
    # print(f"Saliency map (first 5 values): {saliency.flatten()[:5]}")  # Debugging: Print first 5 values

    # Aggregate saliency map along the embedding dimension
    saliency_aggregated = saliency.sum(dim=-1)  # Shape: [batch_size, seq_len]
    # print(f"Aggregated saliency map shape: {saliency_aggregated.shape}")

    return saliency_aggregated.detach().cpu().numpy()

# Define the Model Parameter Randomisation Test (MPRT) metric
metric_randomisation = quantus.MPRT(
    abs=True, 
    return_aggregate=True,
    return_average_correlation=True,
    aggregate_func=custom_aggregation
)

In [17]:
# # Combine inputs into a single tensor for x_batch
# x_batch = torch.cat(
#     [input_ids.float(), attention_mask.float(), sentiment_features.float()], dim=-1
# ).detach().cpu().numpy()  # Shape: (batch_size, total_features)

# # Set y_batch
# y_batch = labels.detach().cpu().numpy()

# # Debugging: Check x_batch and y_batch
# print(f"x_batch shape: {x_batch.shape}")
# print(f"y_batch shape: {y_batch.shape}")
# print(f"y_batch: {y_batch}")

# Ensure correct data types
# x_batch = torch.tensor(x_batch, dtype=torch.float32).to(device)
# y_batch = torch.tensor(y_batch, dtype=torch.int64).to(device)

# Generate saliency maps (only once)
# print("Generating saliency maps using Integrated Gradients...")
# ig = IntegratedGradients(wrapped_model)

# # Get embeddings from embedding layer
# input_embeddings = proto.LLM.embeddings(input_ids)  # Shape: [batch_size, seq_len, hidden_dim]
# input_embeddings.requires_grad_()  # Needed for IG

# # Compute saliency maps
# saliency = ig.attribute(
#     inputs=input_embeddings,
#     additional_forward_args=(attention_mask, sentiment_features),
#     target=y_batch,
#     n_steps=10,
#     internal_batch_size=4
# )

# Convert saliency to numpy
# explanations = saliency.detach().cpu().numpy()

# Aggregate saliency maps to match the dimensionality of x_batch
# explanations = explanations.sum(axis=-1)  # Shape: [batch_size, seq_len]

# # Pad explanations to match the shape of x_batch
# seq_len = input_ids.shape[1]  # Length of the input sequence
# total_features = x_batch.shape[1]  # Total features in x_batch
# padding = total_features - seq_len  # Calculate the padding needed

# # Pad the saliency map with zeros along the last axis
# explanations_padded = np.pad(explanations, ((0, 0), (0, padding)), mode='constant')  # Shape: [batch_size, total_features]

# Replace NaN values in explanations
# explanations_padded = np.nan_to_num(explanations_padded, nan=0.0, posinf=1e6, neginf=-1e6)

# Ensure the shapes match
# assert explanations_padded.shape == x_batch.shape, \
#     f"Shape mismatch: explanations_padded shape {explanations_padded.shape} does not match x_batch shape {x_batch.shape}"

# # Evaluate the Complexity metric
# print("Evaluating Complexity metric...")
# complexity_metric = quantus.Complexity(return_aggregate=True)
# complexity_score = complexity_metric(
#     model=wrapped_model,
#     x_batch=x_batch.cpu().numpy(),
#     y_batch=y_batch.cpu().numpy(),
#     a_batch=explanations_padded,  # Use the padded saliency map
#     device=device
# )
# print(f"Complexity Score: {complexity_score[0]:.4f}")

In [18]:
def explain_func_wrapper(model, inputs, targets, **kwargs):
    """
    Wrapper function to adapt `generate_saliency_map_` for Quantus.
    
    Args:
        model: The wrapped model.
        inputs: Combined input tensor (input_ids, attention_mask, sentiment_features).
        targets: Target labels for the batch.
        kwargs: Additional arguments (not used here).
    
    Returns:
        Numpy array of saliency maps.
    """
    # Extract components from the combined inputs tensor
    seq_len = model.model.max_seq_length  # Sequence length
    sentiment_dim = 4  # Number of sentiment features

    input_ids = inputs[:, :seq_len] #.long()  # Extract input_ids
    attention_mask = inputs[:, seq_len:2 * seq_len]#.long()  # Extract attention_mask
    sentiment_features = inputs[:, 2 * seq_len:] #.float()  # Extract sentiment_features

    # Call the original `generate_saliency_map_` function
    return generate_saliency_map_(
        model=model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        sentiment_features=sentiment_features,
        targets=targets
    )

In [24]:
dummy_x_batch = np.random.rand(32, 204).astype(np.float32)
dummy_y_batch = np.random.randint(0, 10, size=(32,))
dummy_a_batch = np.random.rand(32, 204).astype(np.float32)

try:
    randomisation_score = metric_randomisation(
        model=wrapped_model,
        x_batch=dummy_x_batch,
        y_batch=dummy_y_batch,
        a_batch=dummy_a_batch,
        device=device,
        explain_func=explain_func_wrapper
    )
    print("Randomisation score:", randomisation_score)
except Exception as e:
    print("Error with dummy inputs:", e)

Error with dummy inputs: The dimensions of attribution and input per sample should correspond in either the first or last dimensions, but got shapes [100] and [204]


In [19]:
# Evaluate the Randomisation metric with a loading bar
print("Evaluating Randomisation metric...")
randomisation_score = metric_randomisation(
        model=wrapped_model,
        x_batch=x_batch,
        y_batch=y_batch,
        a_batch=explanations_padded,  # Reuse the same saliency map
        device=device,
        explain_func=explain_func_wrapper  # Use the wrapper function
    )    

# Output the aggregated randomisation score
print("\nFinal Aggregated Randomisation Score:", randomisation_score)
mean_randomisation_score = np.mean(randomisation_score)
print(f"Mean Randomisation Score: {mean_randomisation_score:.3f}")

Evaluating Randomisation metric...


c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\captum\attr\_utils\batching.py:51: UserWarning: Internal batch size cannot be less than the number of input examples. Defaulting to internal batch size of 32 equal to the number of examples.
  warnings.warn(


AssertionError: The dimensions of attribution and input per sample should correspond in either the first or last dimensions, but got shapes [100] and [204]

In [ ]:


# # Define the Model Parameter Randomisation Test (MPRT) metric
# metric_randomisation = quantus.MPRT(
#     abs=True, 
#     return_aggregate=True,
#     return_average_correlation=True,
#     aggregate_func=custom_aggregation
# )

# # Fetch a batch of data from the test dataloader
# print("Preparing test data loader...")
# test_loader = drug_review_dm.test_dataloader()
# batch = next(iter(test_loader))

# # Move data to the appropriate device
# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)
# sentiment_features = batch["sentiment_features"].to(device)
# labels = batch["labels"].to(device)

# # Combine inputs into a single tensor for x_batch
# x_batch = torch.cat(
#     [input_ids.float(), attention_mask.float(), sentiment_features.float()], dim=-1
# ).detach().cpu().numpy()  # Shape: (batch_size, total_features)

# # Set y_batch
# y_batch = labels.detach().cpu().numpy()

# # Debugging: Check x_batch and y_batch
# print(f"x_batch shape: {x_batch.shape}")
# print(f"y_batch shape: {y_batch.shape}")
# print(f"y_batch: {y_batch}")

# # Ensure correct data types
# x_batch = torch.tensor(x_batch, dtype=torch.float32).to(device)
# y_batch = torch.tensor(y_batch, dtype=torch.int64).to(device)


# # Split x_batch back into its components
# seq_len = input_ids.shape[1]
# sentiment_dim = sentiment_features.shape[1]

# input_ids_split = x_batch[:, :seq_len].long()
# attention_mask_split = x_batch[:, seq_len:2 * seq_len].long()
# sentiment_features_split = x_batch[:, 2 * seq_len:].float()

# # Generate saliency maps
# a_batch = generate_saliency_map_(
#     wrapped_model,
#     input_ids_split.to(device),
#     attention_mask_split.to(device),
#     sentiment_features_split.to(device),
#     y_batch
# )

# # Debugging: Check a_batch shape
# print(f"a_batch shape after generation: {a_batch.shape}")

# a_batch = np.nan_to_num(a_batch, nan=0.0, posinf=1e6, neginf=-1e6)

# assert a_batch.shape == x_batch.shape, \
#     f"Shape mismatch: a_batch shape {a_batch.shape} does not match x_batch shape {x_batch.shape}"

# # Evaluate the randomisation score
# randomisation_score = metric_randomisation(
#     model=wrapped_model,  
#     x_batch=x_batch.cpu().numpy(),  
#     y_batch=y_batch.cpu().numpy(),  
#     a_batch=np.array(a_batch),  
#     device=device,  
#     explain_func=generate_saliency_map_,
# )

# # Output the aggregated randomisation score
# print("\nFinal Aggregated Randomisation Score:", randomisation_score)
# mean_randomisation_score = np.mean(randomisation_score)
# print(f"Mean Randomisation Score: {mean_randomisation_score:.3f}")

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# from captum.attr import IntegratedGradients
# import torch
# import numpy as np

# def explain_func(model, inputs, targets, **kwargs):
#     """
#     Explain function for generating attributions using Integrated Gradients.

#     Args:
#         model: The ProtoLM model.
#         inputs: Combined input tensor (input_ids, attention_mask, sentiment_features).
#         targets: Target labels for the batch.
#         kwargs: Additional arguments (not used here).

#     Returns:
#         Numpy array of attributions.
#     """
#     # Ensure inputs is a PyTorch tensor on the correct device
#     if not isinstance(inputs, torch.Tensor):
#         inputs = torch.tensor(inputs, dtype=torch.float32).to(device)
#     else:
#         inputs = inputs.to(device)

#     # Replace invalid values in inputs
#     inputs = torch.nan_to_num(inputs, nan=0.0, posinf=1e6, neginf=-1e6)

#     # Validate inputs shape
#     seq_length = model.max_seq_length
#     sentiment_dim = 4  # Number of sentiment features
#     assert inputs.shape[1] >= seq_length + seq_length + sentiment_dim, \
#         f"Inputs shape is invalid. Expected at least {seq_length + seq_length + sentiment_dim} columns, got {inputs.shape[1]}."

#     # Split the combined inputs back into components
#     input_ids = inputs[:, :seq_length].long()  # Convert to LongTensor
#     attention_mask = inputs[:, seq_length:2 * seq_length]
#     sentiment_features = inputs[:, 2 * seq_length:]

#     # Validate targets
#     assert targets.min() >= 0 and targets.max() < model.num_classes, \
#         f"Targets are out of range. Expected values between 0 and {model.num_classes - 1}, got {targets}."

#     # Define a forward function compatible with IntegratedGradients
#     def forward_func(input_ids, attention_mask, sentiment_features):
#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             sentiment_features=sentiment_features,
#         )
#         return outputs["logits"]

#     # Generate embeddings
#     inputs_embeds = model.LLM.embeddings(input_ids).detach()
#     inputs_embeds.requires_grad_()

#     # Compute attributions using Integrated Gradients
#     ig = IntegratedGradients(forward_func)
#     attributions = ig.attribute(
#         inputs=inputs_embeds,
#         additional_forward_args=(attention_mask, sentiment_features),
#         target=targets,
#     )

#     return attributions.detach().cpu().numpy()


# # Robustness Metric
# try:
#     # Ensure x_batch and y_batch are properly prepared
#     x_batch = torch.cat(
#         [batch["input_ids"].float(), batch["attention_mask"].float(), batch["sentiment_features"].float()],
#         dim=-1
#     ).detach().cpu().numpy()
#     y_batch = batch["labels"].detach().cpu().numpy()

#     robustness_metric = quantus.LocalLipschitzEstimate(
#         nr_samples=15,
#         perturb_std=0.4,
#         perturb_mean=0.0,
#         norm_numerator=quantus.similarity_func.distance_euclidean,
#         norm_denominator=quantus.similarity_func.distance_euclidean,
#         perturb_func=quantus.perturb_func.gaussian_noise,
#         similarity_func=quantus.similarity_func.lipschitz_constant
#     )
#     robustness_score = robustness_metric(
#         model=proto,
#         x_batch=x_batch,
#         y_batch=y_batch,
#         a_batch=explanations_padded,
#         device=device,
#         explain_func=explain_func,
#     )
#     print(f"Robustness Score: {robustness_score}")
# except Exception as e:
#     import traceback
#     traceback.print_exc()

In [ ]:
# ## Step 1: Define a forward_func for Quantus
# def forward_func(embeddings, attention_mask=None, token_type_ids=None, sentiment_features=None):
#     output = proto(
#         inputs_embeds=embeddings,
#         attention_mask=attention_mask,
#         token_type_ids=token_type_ids,
#         sentiment_features=sentiment_features
#     )
#     return output['logits']  # Extract logits from the output dictionary



# ##  Step 2: Choose a batch of test data
# from torch.utils.data import DataLoader

# print("Preparing test data loader...")

# test_loader = drug_review_dm.test_dataloader()
# batch = next(iter(test_loader))
# input_ids = batch['input_ids'].to(device)
# attention_mask = batch['attention_mask'].to(device)
# token_type_ids = batch.get('token_type_ids', None)
# if token_type_ids is not None:
#     token_type_ids = token_type_ids.to(device)
# labels = batch['labels'].to(device)


# ## step 3: Generate Saliency Maps
# from captum.attr import IntegratedGradients

# print("Generating saliency maps using Integrated Gradients...")
# ig = IntegratedGradients(forward_func)

# # Get embeddings from embedding layer
# input_embeddings = proto.LLM.embeddings(input_ids)  # Shape: [batch_size, seq_len, hidden_dim]
# input_embeddings.requires_grad_()  # Needed for IG

# # Now call IntegratedGradients
# saliency = ig.attribute(
#     inputs=input_embeddings,
#     additional_forward_args=(attention_mask, token_type_ids, batch['sentiment_features'].to(device)),
#     target=0,
#     n_steps=10,
#     internal_batch_size=4
# )


# # # Compute saliency for class 0 for simplicity
# # saliency = ig.attribute(inputs=input_ids, 
# #                         additional_forward_args=(attention_mask, token_type_ids), 
# #                         target=0,  # You could loop over all targets too
# #                         n_steps=50)


# ## Step 4: Format inputs for Quantus
# # Convert token-level saliency to shape [batch_size, seq_len]
# print("Formatting inputs for Quantus...")
# explanations = saliency.detach().cpu().numpy()
# inputs_np = input_ids.detach().cpu().numpy()
# labels_np = labels.detach().cpu().numpy()

# ## step 5: Compute Metrics
# import quantus

# metrics = {
#     "Complexity": quantus.Complexity(),
#     "Faithfulness": quantus.FaithfulnessCorrelation(),
#     "Robustness": quantus.LocalLipschitzEstimate(),
#     "Sensitivity": quantus.SensitivityN()
# }

# # Combine inputs into a single tensor for Quantus
# # Extract inputs from the batch
# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)
# sentiment_features = batch["sentiment_features"].to(device)
# labels = batch["labels"].to(device)
# x_batch = torch.cat(
#     [input_ids.float(), attention_mask.float(), sentiment_features.float()], dim=-1
# ).detach().cpu().numpy()  # Shape: (batch_size, total_features)
# y_batch = labels.detach().cpu().numpy()

# print("Computing metrics...")
# for name, metric in metrics.items():
#     print(f"Computing {name} metric...")
#     score = metric(
#         x_batch=x_batch,
#         y_batch=y_batch,
#         explanations=explanations,
#         inputs=inputs_np,
#         model=proto,
#         targets=labels_np,
#         task="classification",  # Important!
#         device=device,
#         forward_func=forward_func
#     )
#     print(f"{name}: {score}")


In [ ]:
# import torch
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# from captum.attr import IntegratedGradients
# import quantus

# # Your Proto-LM model
# model = proto  # make sure this is already loaded and moved to the correct device
# model_name_for_csv = f"ProtoLM_{model_name}"
# model.eval()

# # Wrapper for Quantus-compatible interface
# import torch.nn as nn

# class ModelWrapper(nn.Module):
#     def __init__(self, model):
#         super(ModelWrapper, self).__init__()
#         self.model = model

#     def forward(self, input_ids=None, attention_mask=None, sentiment_features=None, inputs_embeds=None):
#         device = next(self.model.parameters()).device

#         # Ensure inputs_embeds is passed correctly
#         if inputs_embeds is not None:
#             inputs_embeds = inputs_embeds.to(device)
#             input_ids = None  # Avoid passing input_ids when using embeddings
#         elif input_ids is not None:
#             input_ids = input_ids.to(device).long()

#         if attention_mask is not None:
#             attention_mask = attention_mask.to(device)
#         if sentiment_features is not None:
#             sentiment_features = sentiment_features.to(device)

#         outputs = self.model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,  # Ensure this is passed
#             sentiment_features=sentiment_features,
#             inputs_embeds=inputs_embeds  # Pass inputs_embeds explicitly
#         )
#         return outputs['logits']


# wrapped_model = ModelWrapper(model)
# # Debugging: Add print statements to verify shapes
# print("Fetching batch...")
# batch = next(iter(drug_review_dm.test_dataloader()))
# device = next(model.parameters()).device
# print(f"Running on device: {device}")

# # Extract inputs from the batch
# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)
# sentiment_features = batch["sentiment_features"].to(device)
# labels = batch["labels"].to(device)

# # Debugging: Print shapes of inputs
# print(f"input_ids shape: {input_ids.shape}")  # Expected: (batch_size, seq_length)
# print(f"attention_mask shape: {attention_mask.shape}")  # Expected: (batch_size, seq_length)
# print(f"sentiment_features shape: {sentiment_features.shape}")  # Expected: (batch_size, num_features)
# print(f"labels shape: {labels.shape}")  # Expected: (batch_size,)

# # Combine inputs into a single tensor for Quantus
# x_batch = torch.cat(
#     [input_ids.float(), attention_mask.float(), sentiment_features.float()], dim=-1
# ).detach().cpu().numpy()  # Shape: (batch_size, total_features)
# y_batch = labels.detach().cpu().numpy()

# # Define Quantus metrics
# metrics = {
#     "Robustness": quantus.LocalLipschitzEstimate(),
#     # Add other metrics as needed
# }

# # Initialize IntegratedGradients
# ig = IntegratedGradients(wrapped_model)

# # Define a proper explain function for Quantus

# def explain_func(model, inputs, targets, **kwargs):
#     # Debugging: Print shape of inputs
#     print(f"inputs shape in explain_func: {inputs.shape}")  # Expected: (batch_size, total_features)

#     # Split the combined inputs back into components
#     seq_length = input_ids.shape[-1]
#     sentiment_dim = sentiment_features.shape[-1]

#     input_ids_t = torch.tensor(inputs[:, :seq_length]).long().to(device)
#     attention_mask_t = torch.tensor(inputs[:, seq_length:2 * seq_length]).to(device)
#     sentiment_features_t = torch.tensor(inputs[:, 2 * seq_length:]).to(device)

#     # Debugging: Print shapes after splitting
#     print(f"input_ids_t shape: {input_ids_t.shape}")  # Expected: (batch_size, seq_length)
#     print(f"attention_mask_t shape: {attention_mask_t.shape}")  # Expected: (batch_size, seq_length)
#     print(f"sentiment_features_t shape: {sentiment_features_t.shape}")  # Expected: (batch_size, sentiment_dim)

#     # Generate embeddings
#     inputs_embeds = model.model.LLM.embeddings(input_ids_t).detach().requires_grad_()

#     # Debugging: Print shape of inputs_embeds
#     print(f"inputs_embeds shape: {inputs_embeds.shape}")  # Expected: (batch_size, seq_length, hidden_size)

#     # Compute attributions using Integrated Gradients
#     attributions = ig.attribute(
#         inputs=inputs_embeds,
#         additional_forward_args=(attention_mask_t, sentiment_features_t),
#         target=targets,
#     )
#     return attributions.detach().cpu().numpy()

# # Evaluate metrics
# results = {}
# print("Evaluating Quantus metrics...")
# for name, metric in tqdm(metrics.items()):
#     print(f"Running metric: {name}")
#     try:
#         results[name] = metric(
#             model=wrapped_model,
#             x_batch=x_batch,
#             y_batch=y_batch,
#             explain_func=explain_func,
#             batch_size=16,  # Adjust batch size as needed
#         )
#         print(f"{name} score: {results[name]}")
#     except Exception as e:
#         print(f"Error while running metric {name}: {e}")
#         raise

# # Save results to CSV
# print("Saving results to CSV...")
# results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Score"])
# results_df["Model"] = model_name_for_csv
# results_df.reset_index(inplace=True)
# results_df.rename(columns={"index": "Metric"}, inplace=True)

# csv_filename = f"quantus_metrics_{model_name_for_csv}.csv"
# results_df.to_csv(csv_filename, index=False)
# print(f"Quantus results saved to {csv_filename}")

### Evalute the model

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
from tqdm import tqdm  # Import tqdm for the loading bar

# Function to calculate MSE, MAE, and RMSE
def calculate_metrics(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing Batches", leave=False):  # Add tqdm for progress bar
            # Move data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Pass only input_ids and attention_mask to the model
            # outputs = model(
            #     input_ids=input_ids,
            #     attention_mask=attention_mask,
            #     sentiment_features=None  # Exclude sentiment features
            # )

            # print(f"sentiment_neg shape: {batch['sentiment_neg'].shape}")
            # print(f"sentiment_neu shape: {batch['sentiment_neu'].shape}")
            # print(f"sentiment_pos shape: {batch['sentiment_pos'].shape}")
            # print(f"sentiment_compound shape: {batch['sentiment_compound'].shape}")

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                sentiment_features = torch.cat((
                    batch["sentiment_neg"].unsqueeze(-1),
                    batch["sentiment_neu"].unsqueeze(-1),
                    batch["sentiment_pos"].unsqueeze(-1),
                    batch["sentiment_compound"].unsqueeze(-1)
                ), dim=1)
            )
            preds = torch.argmax(outputs["logits"], dim=1)  # Predicted class

            # Collect predictions and labels
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Flatten predictions and labels
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Calculate metrics
    mse = mean_squared_error(all_labels, all_preds)
    mae = mean_absolute_error(all_labels, all_preds)
    rmse = np.sqrt(mse)

    return mse, mae, rmse

# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
proto.to(device)

# Calculate metrics for train, validation, and test sets
datasets = {
    "Train": drug_review_dm.train_dataloader(),
    "Validation": drug_review_dm.val_dataloader(),
    "Test": drug_review_dm.test_dataloader()
}

results = []

for dataset_name, dataloader in datasets.items():
    print(f"Calculating metrics for {dataset_name}...")
    mse, mae, rmse = calculate_metrics(proto, dataloader, device)
    print(f"{dataset_name} Metrics:")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    results.append({"Dataset": dataset_name, "MSE": mse, "MAE": mae, "RMSE": rmse})

# Export results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv("metrics_results.csv", index=False)
print("Metrics saved to 'metrics_results.csv'")

### Plot similarity between test set cases and the learned concepts similar to figure 3 in their paper

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Get prototype vectors (concepts)
prototypes = proto.prototypes.detach().cpu().numpy()  # shape: (num_prototypes, hidden_dim)

# Get representations for some samples (e.g., from your test set)
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    # Get last hidden states for the batch
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()  # [CLS] token or mean pooling



# prototypes: (num_prototypes, hidden_dim)
# sample_reps: (batch_size, hidden_dim)
# Assume proto.prototype_class_vec exists and is (num_prototypes, num_classes)

# 1. Identify positive and negative prototypes
# If proto.prototype_class_vec is one-hot or softmax over classes:
proto_class = proto.prototype_class_vec.detach().cpu().numpy()  # (num_prototypes, num_classes)
positive_proto_idx = np.argmax(proto_class[:, 1])  # class 1 = positive
negative_proto_idx = np.argmax(proto_class[:, 0])  # class 0 = negative

positive_proto = torch.tensor(prototypes[positive_proto_idx])
negative_proto = torch.tensor(prototypes[negative_proto_idx])

# 2. Compute similarities for each sample
sample_vecs = torch.tensor(sample_reps)  # (batch_size, hidden_dim)
sim_pos = F.cosine_similarity(sample_vecs, positive_proto.unsqueeze(0), dim=1)
sim_neg = F.cosine_similarity(sample_vecs, negative_proto.unsqueeze(0), dim=1)

# 3. Get ground-truth labels for the batch
labels = batch['labels'].cpu().numpy()

# 4. Plot
plt.figure(figsize=(8, 8))
for label in np.unique(labels):
    idxs = np.where(labels == label)[0]
    plt.scatter(sim_pos[idxs], sim_neg[idxs], label=f"Class {label}", alpha=0.7)
plt.xlabel("Similarity to Positive Prototype")
plt.ylabel("Similarity to Negative Prototype")
plt.title("2D Prototypical Space (like Proto-LM Fig. 3)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

sample_vec = sample_reps[0]  # pick one sample
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F

# 1. Get all review texts and their embeddings from the training set
train_dataset = drug_review_dm.dataset["train"]
all_texts = train_dataset["review"]

# Get all input_ids and attention_mask for the train set
input_ids = train_dataset["input_ids"]
attention_mask = train_dataset["attention_mask"]

# Compute all embeddings (CLS token)
all_embeddings = []
batch_size = 128
for i in range(0, len(input_ids), batch_size):
    batch_input_ids = input_ids[i:i+batch_size].to(proto.device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(proto.device)
    with torch.no_grad():
        outputs = proto.LLM(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            output_hidden_states=True
        )
        batch_embeds = outputs.hidden_states[-1][:, 0, :].cpu()  # CLS token
        all_embeddings.append(batch_embeds)
all_embeddings = torch.cat(all_embeddings, dim=0)  # (num_samples, hidden_dim)

# 2. For each prototype, find the closest text
prototypes = proto.prototypes.detach().cpu()  # (num_prototypes, hidden_dim)
closest_texts = []
for proto_vec in prototypes:
    sims = F.cosine_similarity(all_embeddings, proto_vec.unsqueeze(0), dim=1)
    idx = torch.argmax(sims).item()
    closest_texts.append(all_texts[idx])

# 3. Save to CSV
df = pd.DataFrame({"prototype_index": range(len(closest_texts)), "closest_text": closest_texts})
df.to_csv("prototype_texts.csv", index=False)